# Add the LPS metrics to the curr table
1. mean purity metrics
2. major lps motif (the one that comes up in the most alleles in the population based on AC)
3. number of lps motifs
3. mean LPS
4. median LPS
5. LPS standard deviation

In [1]:
import tdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyranges as pr
from sklearn.decomposition import PCA
import seaborn as sb
import sys
import trf
from scipy.stats import pearsonr
import matplotlib.colors as mcolors
import functools
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from scipy.stats import kruskal
from get_lps_len_motif import *
from Levenshtein import distance as edit_distance
from collections import deque
import regex as re
# import my own modules
sys.path.insert(0, "../QC_scripts")
from QC_utils import *  # my QC functions
sys.path.insert(0, "..")
from analysis_utils import *  # tdb functions for downstream analysis

# Test and get all LPS and Revised Purity For Our Set and Set Up New Off Target Filter
unfortunately I did not save the allele frequency for my tdb :( this means I need to figure

In [2]:
def get_AF_from_observed(observed_lengths):
    """
    Get the allele frequency from the observed lengths
    :param observed_lengths: list of observed lengths
    :return: list of allele frequencies
    """
    lcnts = observed_lengths["LocusID"].value_counts()
    acnts = (observed_lengths.groupby(["LocusID"])["allele_number"]
             .value_counts()
             .rename("AC")
             .reset_index(level=1))
    acnts['AF'] = acnts['AC'] / lcnts
    return acnts.reset_index()
def get_lps_len_motif_no_homopolymers(allele, refMotifs, maxKmerLen=20):
    """
    Get the length and motif of the longest perfect repeat (LPS) in a given allele and a given set of motifs
    :param allele: str, the allele sequence
    :param refMotifs: list, the reference motifs for this locus according to the catalog
    :return: tuple, (length of the LPS, simplified LPS motif, original motif of the LPS) (modifed so I can have the original motif for purity)
    """
    approvedMotifs={}
    approvedMotifCounts={}
    lpsLength=0
    lpsMotif=''
    spanCovered=0
    done=0
    topLPSLength=0
    topLPSMotif=''
    for k in refMotifs: # this is from the list of reference motifs
        # if any one of the reference motifs has an LPS greater than half the repeat span, that is the LPS
        matches=re.findall('(?:' + k + ')+',allele)
        if len(matches)>0 and len(max(matches))>0.5*len(allele):
            lpsLength = len(max(matches))
            lpsMotif = SequenceAttributes.get_simple_motif(k)
            topLPSMotif = k
            done=1
            break
        else:
            # if not, track the spans covered by this motif
            spanCovered+=sum([len(x) for x in matches])
            if len(matches)>0 and len(max(matches))>topLPSLength:
                topLPSLength=len(max(matches))
                topLPSMotif=k
    if done==0: # this is reached if no reference motif covers more than half the repeat
        # is there enough space left in the repeat for a sequence to be the LPS that isn't already accounted for?
        if (len(allele)-spanCovered)>topLPSLength:
            # do a k-mer search for novel motifs
            if len(allele)<3*maxKmerLen:
                maxKmerLen=int(len(allele)/3)
            for k in range(2, maxKmerLen + 1):  # changed from 1 to 2, we don't want to return homopolymers
                if done == 1:
                    break
                words = re.findall('.' * k, allele)
                words = ' '.join(words)
                matches = set(re.findall(r"\b(\w+)\s+\1\b\s\1\b", words))  # only keep k-mers with at least 3 consecutive matches
                for motif in matches:
                    # Skip motifs that are composed of only one unique character
                    if len(set(motif)) < 2:
                        continue
                    matches = re.findall('(?:' + motif + ')+', allele)
                    spanCovered += sum([len(x) for x in matches])
                    if len(max(matches)) > 0.5 * len(allele):
                        topLPSLength = len(max(matches))
                        topLPSMotif = motif
                        done = 1
                        break
                    elif len(max(matches)) > topLPSLength:
                        topLPSLength = len(max(matches))
                        topLPSMotif = motif
        # pick the best LPS
        lpsLength = topLPSLength
        lpsMotif = SequenceAttributes.get_simple_motif(topLPSMotif)

    return lpsLength, lpsMotif, topLPSMotif

In [3]:
def get_aggregate_motifs(allele_table_full_df, AC):
    """
    Get (1) the motif that is the most common in the population based on the sum of allele counts when grouped by simplified_lps_motif
    and (2) a list of all the motifs that are present in the population according to simplified_lps_motif
    """
    # NEW 02182026: DO NOT COUNT NULL MOTIFS IN THE MOST COMMON MOTIF CALCULATION, BECAUSE THEY ARE NOT REAL MOTIFS
    allele_table_full_df_non_null = allele_table_full_df[allele_table_full_df["simple_motif_lps"].isna() == False]
    # Get the most common motif in the population
    most_common_motif = allele_table_full_df_non_null.groupby("simple_motif_lps")["allele_number"].sum().idxmax()
    # Get the list of all motifs in the population
    all_motifs = allele_table_full_df_non_null["simple_motif_lps"].unique()
    return most_common_motif, all_motifs

In [4]:
curr = pd.read_csv("final_manuscript_labels.tsv", sep="\t")
allele_table_file = "/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/daTR_update_10spanning.csv" # saved out allele table
allele_table = pd.read_csv(allele_table_file)
allele_table

/tmp/ipykernel_2365453/4102131060.py:1: DtypeWarning: Columns (3,9) have mixed types. Specify dtype option on import or set low_memory=False.
  curr = pd.read_csv("final_manuscript_labels.tsv", sep="\t")


,LocusID,allele_number,allele_length,sequence,original_LocusID,span_number,LocusID:allele_number
0,15,0,13.0,b'ACCTCCCTCCCTC',15,NaN,15:0
1,16,0,12.0,b'TCTGCTGCTGCT',16,NaN,16:0
2,16,1,12.0,b'TCTGCTGCTTCT',16,NaN,16:1
3,16,2,12.0,b'TCTGCTTCTTCT',16,NaN,16:2
4,17,0,14.0,b'CGCTGCTGCTGCTG',17,NaN,17:0
...,...,...,...,...,...,...,...
4221944,4053467,5,44.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG',4053467,1.0,4053467:5
4221945,4053467,3,50.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG...,4053467,1.0,4053467:3
4221946,4053467,4,50.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCCGGGCCTGGGCCTG...,4053467,1.0,4053467:4
4221947,4053467,2,56.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG...,4053467,1.0,4053467:2


In [5]:
# read the AC and AF from np file
all_AF_array = np.load("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/daTR_update_10spanning_AF_array.npy")
all_AF= pd.DataFrame(all_AF_array, columns=["LocusID", "allele_number", "AC", "AF"])

In [6]:
# Calculate this for the full curr set and allele table
# Add the str_motif to the allele table using the LocusID
allele_table_full = allele_table.merge(curr[["LocusID", "str_motif", "simple_motif"]], on="LocusID", how="left")
allele_table_full = allele_table_full[~allele_table_full.str_motif.isna()].reset_index(drop=True)
# Calculate LPS for the full allele table
allele_table_full["lps"] = allele_table_full.apply(lambda x: get_lps_len_motif_no_homopolymers(x["sequence"], [x["str_motif"]], maxKmerLen=6), axis=1)

# Convert the LPS results into a DataFrame
allele_table_full_df = pd.DataFrame(allele_table_full["lps"].tolist(), columns=['LPS', 'simple_motif_lps', 'major_kmer'])

# Add the LocusID and length columns to the DataFrame based on the index
allele_table_full_df["LocusID"] = allele_table_full["LocusID"]
allele_table_full_df["allele_length"] = allele_table_full["allele_length"]

# Add the sequence, str_motif, and simple_motif columns to the DataFrame
allele_table_full_df["sequence"] = allele_table_full["sequence"]
allele_table_full_df["str_motif"] = allele_table_full["str_motif"]
allele_table_full_df["simple_motif"] = allele_table_full["simple_motif"]

# Calculate naive purity and purity percentages
allele_table_full_df["naive_purity"] = allele_table_full_df.apply(lambda x: x["LPS"] / x["allele_length"], axis=1)
allele_table_full_df["purity_percent_exact"] = allele_table_full_df.apply(lambda x: get_purity_lps_motif(x["major_kmer"], x["sequence"]), axis=1)
allele_table_full_df["purity_percent_edit"] = allele_table_full_df.apply(lambda x: get_purity_lps_edit(x["sequence"], x["major_kmer"]), axis=1)


allele_table_full_df = allele_table_full_df.merge(allele_table[["LocusID", "sequence", "allele_number"]], on=["LocusID", "sequence"], how="left")

allele_table_full_df


,LPS,simple_motif_lps,major_kmer,LocusID,allele_length,sequence,str_motif,simple_motif,naive_purity,purity_percent_exact,purity_percent_edit,allele_number
0,12,AGGG,CCTC,15,13.0,b'ACCTCCCTCCCTC',CCTC,AGGG,0.923077,0.750000,0.687500,0
1,9,AGC,CTG,16,12.0,b'TCTGCTGCTGCT',CTG,AGC,0.750000,0.600000,0.733333,0
2,6,AGC,CTG,16,12.0,b'TCTGCTGCTTCT',CTG,AGC,0.500000,0.400000,0.666667,1
3,3,AGC,CTG,16,12.0,b'TCTGCTTCTTCT',CTG,AGC,0.250000,0.200000,0.600000,2
4,12,AGC,GCT,17,14.0,b'CGCTGCTGCTGCTG',GCT,AGC,0.857143,0.705882,0.764706,0
...,...,...,...,...,...,...,...,...,...,...,...,...
3700428,36,AGGCCC,GGCCTG,4053467,44.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG',GGCCTG,AGGCCC,0.818182,0.765957,0.829787,5
3700429,42,AGGCCC,GGCCTG,4053467,50.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG...,GGCCTG,AGGCCC,0.840000,0.792453,0.849057,3
3700430,18,AGGCCC,GGCCTG,4053467,50.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCCGGGCCTGGGCCTG...,GGCCTG,AGGCCC,0.360000,0.679245,0.830189,4
3700431,48,AGGCCC,GGCCTG,4053467,56.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG...,GGCCTG,AGGCCC,0.857143,0.813559,0.864407,2


In [7]:
allele_table_full_df = allele_table_full_df.merge(allele_table[["LocusID", "sequence"]], on=["LocusID", "sequence"], how="left")

In [8]:
# Only include the alleles that are observed in the population in the allele table otherwise we are including lps that are not in the population
# get the number of unique locusid allele pairs
unique_locusid_allele_pairs = allele_table_full_df[allele_table_full_df.LocusID.isin(curr.LocusID)][["LocusID", "allele_number"]].drop_duplicates()
# get the same for the AC
unique_locusid_allele_pairs_AC = all_AF[all_AF.LocusID.isin(curr.LocusID)][["LocusID", "allele_number"]].drop_duplicates()
print(unique_locusid_allele_pairs.shape)
print(unique_locusid_allele_pairs_AC.shape)

final_allele_table = allele_table_full_df[allele_table_full_df.LocusID.isin(curr.LocusID)].drop_duplicates()
final_allele_table # there are some alleles that are unique by sequence but not by length so there are some duplicates

(3700433, 2)
(3700433, 2)


,LPS,simple_motif_lps,major_kmer,LocusID,allele_length,sequence,str_motif,simple_motif,naive_purity,purity_percent_exact,purity_percent_edit,allele_number
0,12,AGGG,CCTC,15,13.0,b'ACCTCCCTCCCTC',CCTC,AGGG,0.923077,0.750000,0.687500,0
1,9,AGC,CTG,16,12.0,b'TCTGCTGCTGCT',CTG,AGC,0.750000,0.600000,0.733333,0
2,6,AGC,CTG,16,12.0,b'TCTGCTGCTTCT',CTG,AGC,0.500000,0.400000,0.666667,1
3,3,AGC,CTG,16,12.0,b'TCTGCTTCTTCT',CTG,AGC,0.250000,0.200000,0.600000,2
4,12,AGC,GCT,17,14.0,b'CGCTGCTGCTGCTG',GCT,AGC,0.857143,0.705882,0.764706,0
...,...,...,...,...,...,...,...,...,...,...,...,...
3700428,36,AGGCCC,GGCCTG,4053467,44.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG',GGCCTG,AGGCCC,0.818182,0.765957,0.829787,5
3700429,42,AGGCCC,GGCCTG,4053467,50.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG...,GGCCTG,AGGCCC,0.840000,0.792453,0.849057,3
3700430,18,AGGCCC,GGCCTG,4053467,50.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCCGGGCCTGGGCCTG...,GGCCTG,AGGCCC,0.360000,0.679245,0.830189,4
3700431,48,AGGCCC,GGCCTG,4053467,56.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG...,GGCCTG,AGGCCC,0.857143,0.813559,0.864407,2


# FIXME these need to be weighted by the AF

In [9]:
final_allele_table = final_allele_table.merge(all_AF[["LocusID", "allele_number", "AC"]], on=["LocusID", "allele_number"], how="left")


In [10]:
import numpy as np
from statsmodels.stats.weightstats import DescrStatsW
def weighted_mean(df, values, weights, groupby):
    df = df.copy()
    grouped = df.groupby(groupby)
    df['weighted_average'] = df[values] / grouped[weights].transform('sum') * df[weights]
    return grouped['weighted_average'].sum(min_count=1) 
    
def calculate_weighted_stats(grouped_df, values, weights):
    """
    Calculate weighted statistics for each group in a grouped DataFrame.

    :param grouped_df: Grouped DataFrame
    :param values: Column name for the values to calculate statistics on
    :param weights: Column name for the weights
    :return: DataFrame with weighted mean, median, and standard deviation for each group
    """
    results = []
    for group_name, group in grouped_df:
        weighted_stats = DescrStatsW(group[values], weights=group[weights], ddof=0)
        results.append({
            'group': group_name,
            'weighted_mean': weighted_stats.mean,
            'weighted_std': weighted_stats.std,
            'weighted_var': weighted_stats.var
        })
    return pd.DataFrame(results)


# Get the weighted stats but only include alleles whose simple_lps_motif are the same as the most common

In [11]:
# don't include alleles that don't have an LPS or LPS motif from the beginning
alleles_per_motif = final_allele_table[final_allele_table.simple_motif_lps != "" ][["LocusID","allele_number","simple_motif_lps"]].merge(all_AF, on=["LocusID","allele_number"], how="left").groupby(["LocusID","simple_motif_lps"]).AC.sum()

In [12]:
# don't consider null motifs in the percent and maximum motif calculations because they are alleles that did not have a motif that met the criteria for being the LPS, so they are not really motifs
alleles_no_null = alleles_per_motif.reset_index()[alleles_per_motif.reset_index().simple_motif_lps != ""]
alleles_per_motif_percent = alleles_no_null.merge(alleles_no_null.groupby("LocusID").AC.sum(), on="LocusID", how="left", suffixes=("", "_total"))
alleles_per_motif_percent["percent"] = alleles_per_motif_percent.AC/alleles_per_motif_percent.AC_total
alleles_per_motif_percent

,LocusID,simple_motif_lps,AC,AC_total,percent
0,15,AGGG,143.0,143.0,1.0
1,16,AGC,160.0,160.0,1.0
2,17,AGC,161.0,161.0,1.0
3,18,ACC,161.0,161.0,1.0
4,19,AGG,171.0,171.0,1.0
...,...,...,...,...,...
2383472,4542948,AGG,200.0,200.0,1.0
2383473,4542949,CCG,200.0,200.0,1.0
2383474,4542950,CCG,199.0,199.0,1.0
2383475,4542951,AGC,200.0,200.0,1.0


In [13]:
alleles_per_motif_max = alleles_per_motif.loc[alleles_per_motif.groupby("LocusID").idxmax()]
# get what percentage of the total alleles this is
alleles_per_motif_max = alleles_per_motif_max.reset_index()
print(alleles_per_motif_max.head())

# do the same but record the max percentage of the total alleles instead of the max AC
alleles_per_motif_max_percent = alleles_per_motif_percent.loc[alleles_per_motif_percent.groupby("LocusID").percent.idxmax()]
alleles_per_motif_max_percent = alleles_per_motif_max_percent.reset_index()

# merge the 2 on LocusID and simple_motif_lps and only keep the LocusID, simple_motif_lps, AC, and percent
alleles_per_motif_max = alleles_per_motif_max.merge(alleles_per_motif_max_percent[["LocusID", "simple_motif_lps", "percent"]], on=["LocusID", "simple_motif_lps"], how="left")
alleles_per_motif_max.columns = ["LocusID", "most_common_motif","num_alleles_max_motif", "percent_max_motif"]
alleles_per_motif_max


   LocusID simple_motif_lps     AC
0       15             AGGG  143.0
1       16              AGC  160.0
2       17              AGC  161.0
3       18              ACC  161.0
4       19              AGG  171.0


,LocusID,most_common_motif,num_alleles_max_motif,percent_max_motif
0,15,AGGG,143.0,1.0
1,16,AGC,160.0,1.0
2,17,AGC,161.0,1.0
3,18,ACC,161.0,1.0
4,19,AGG,171.0,1.0
...,...,...,...,...
2380954,4542948,AGG,200.0,1.0
2380955,4542949,CCG,200.0,1.0
2380956,4542950,CCG,199.0,1.0
2380957,4542951,AGC,200.0,1.0


In [14]:
# find the rows where the locusid comes up more than once in alleles_per_motif
number_motifs = alleles_per_motif.reset_index()
# don't include null motifs in the count of number of motifs because they are not real motifs
number_motifs = number_motifs[number_motifs.simple_motif_lps != ""]
number_motifs = number_motifs.groupby("LocusID").count().reset_index()
number_motifs.drop(columns=["AC"], inplace=True)
number_motifs.columns = ["LocusID", "num_total_lps_motifs"]
number_motifs

,LocusID,num_total_lps_motifs
0,15,1
1,16,1
2,17,1
3,18,1
4,19,1
...,...,...
2380954,4542948,1
2380955,4542949,1
2380956,4542950,1
2380957,4542951,1


In [15]:
number_motifs[number_motifs.num_total_lps_motifs > 1].num_total_lps_motifs.value_counts()

num_total_lps_motifs
2    2420
3      39
4       4
5       2
Name: count, dtype: int64

## Calculate the weighted stats for all regions as the weighted stats for just alleles with the major motif

In [16]:
# merge with the final allele table to get the allele information for all loci
final_allele_table_variable_motifs = pd.merge(final_allele_table, alleles_per_motif_max, on=["LocusID"], how="left")
# subset this to only include the alleles that have simlpe_motif_lps equal to the most common motif
final_allele_table_variable_motifs = final_allele_table_variable_motifs[(final_allele_table_variable_motifs.simple_motif_lps == final_allele_table_variable_motifs.most_common_motif)]

final_allele_table_variable_motifs


,LPS,simple_motif_lps,major_kmer,LocusID,allele_length,sequence,str_motif,simple_motif,naive_purity,purity_percent_exact,purity_percent_edit,allele_number,AC,most_common_motif,num_alleles_max_motif,percent_max_motif
0,12,AGGG,CCTC,15,13.0,b'ACCTCCCTCCCTC',CCTC,AGGG,0.923077,0.750000,0.687500,0,143.0,AGGG,143.0,1.0
1,9,AGC,CTG,16,12.0,b'TCTGCTGCTGCT',CTG,AGC,0.750000,0.600000,0.733333,0,29.0,AGC,160.0,1.0
2,6,AGC,CTG,16,12.0,b'TCTGCTGCTTCT',CTG,AGC,0.500000,0.400000,0.666667,1,96.0,AGC,160.0,1.0
3,3,AGC,CTG,16,12.0,b'TCTGCTTCTTCT',CTG,AGC,0.250000,0.200000,0.600000,2,35.0,AGC,160.0,1.0
4,12,AGC,GCT,17,14.0,b'CGCTGCTGCTGCTG',GCT,AGC,0.857143,0.705882,0.764706,0,160.0,AGC,161.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3700428,36,AGGCCC,GGCCTG,4053467,44.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG',GGCCTG,AGGCCC,0.818182,0.765957,0.829787,5,78.0,AGGCCC,200.0,1.0
3700429,42,AGGCCC,GGCCTG,4053467,50.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG...,GGCCTG,AGGCCC,0.840000,0.792453,0.849057,3,5.0,AGGCCC,200.0,1.0
3700430,18,AGGCCC,GGCCTG,4053467,50.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCCGGGCCTGGGCCTG...,GGCCTG,AGGCCC,0.360000,0.679245,0.830189,4,3.0,AGGCCC,200.0,1.0
3700431,48,AGGCCC,GGCCTG,4053467,56.0,b'AGAGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTGGGCCTG...,GGCCTG,AGGCCC,0.857143,0.813559,0.864407,2,18.0,AGGCCC,200.0,1.0


In [17]:
# get the weighted stats
grouped = final_allele_table_variable_motifs.groupby("LocusID")
weighted_stats_variable = calculate_weighted_stats(grouped, values="LPS", weights="AC")

weighted_stats_variable.columns = ["LocusID", "weighted_mean_LPS", "weighted_std_LPS", "weighted_var_LPS"]
weighted_stats_variable

,LocusID,weighted_mean_LPS,weighted_std_LPS,weighted_var_LPS
0,15,12.000000,0.000000,0.000000
1,16,5.887500,1.894028,3.587344
2,17,11.944099,0.707093,0.499981
3,18,8.981366,0.235698,0.055553
4,19,9.000000,0.000000,0.000000
...,...,...,...,...
2380954,4542948,6.345000,0.957066,0.915975
2380955,4542949,9.000000,0.000000,0.000000
2380956,4542950,8.969849,0.299238,0.089543
2380957,4542951,44.640000,36.198901,1310.360400


In [18]:
# merge the motif stats with the curr dataframe on LocusID
# merge alleles_per_motif_max with curr_stats on LocusID
curr_stats = weighted_stats_variable.merge(alleles_per_motif_max, on=["LocusID"], how="left")
# merge number_motifs with curr_stats on LocusID
curr_stats = curr_stats.merge(number_motifs, on=["LocusID"], how="left")
curr_stats.head()

,LocusID,weighted_mean_LPS,weighted_std_LPS,weighted_var_LPS,most_common_motif,num_alleles_max_motif,percent_max_motif,num_total_lps_motifs
0,15,12.000000,0.000000,0.000000,AGGG,143.0,1.0,1
1,16,5.887500,1.894028,3.587344,AGC,160.0,1.0,1
2,17,11.944099,0.707093,0.499981,AGC,161.0,1.0,1
3,18,8.981366,0.235698,0.055553,ACC,161.0,1.0,1
4,19,9.000000,0.000000,0.000000,AGG,171.0,1.0,1


In [19]:
# add the unique_lps so we don't have to do this again later -- we need to not count alleles that have an LPS or LPS motif here since they are invalid alleles
# get the number of unique LPS lengths for each locus not including alleles that have an LPS motif since those are not real LPS
unique_lps = final_allele_table[final_allele_table.simple_motif_lps != ""].groupby("LocusID").LPS.nunique().reset_index()
unique_lps.columns = ["LocusID", "num_unique_lps"]
# merge unique_lps with curr_stats on LocusID
curr_stats = curr_stats.merge(unique_lps, on=["LocusID"], how="left")
curr_stats

,LocusID,weighted_mean_LPS,weighted_std_LPS,weighted_var_LPS,most_common_motif,num_alleles_max_motif,percent_max_motif,num_total_lps_motifs,num_unique_lps
0,15,12.000000,0.000000,0.000000,AGGG,143.0,1.0,1,1
1,16,5.887500,1.894028,3.587344,AGC,160.0,1.0,1,3
2,17,11.944099,0.707093,0.499981,AGC,161.0,1.0,1,2
3,18,8.981366,0.235698,0.055553,ACC,161.0,1.0,1,2
4,19,9.000000,0.000000,0.000000,AGG,171.0,1.0,1,1
...,...,...,...,...,...,...,...,...,...
2380954,4542948,6.345000,0.957066,0.915975,AGG,200.0,1.0,1,2
2380955,4542949,9.000000,0.000000,0.000000,CCG,200.0,1.0,1,1
2380956,4542950,8.969849,0.299238,0.089543,CCG,199.0,1.0,1,2
2380957,4542951,44.640000,36.198901,1310.360400,AGC,200.0,1.0,1,18


In [20]:
# add genomic annotations and disease associations to curr_stats
curr_stats = curr_stats.merge(curr[["LocusID","chrom","start","end","disease_associated",
           "str_period", "str_motif","mei","disease_id","GencodeGeneRegion"]], on="LocusID", how="left")
curr_stats.disease_id.value_counts()

disease_id
non_disease_associated    2380904
SCA37                           1
NIID                            1
FAME2                           1
FRA2A                           1
SPD                             1
GAD                             1
SCA7                            1
FAME4                           1
HD                              1
CANVAS                          1
CCHS                            1
FAME7                           1
FAME3                           1
SCA12                           1
SCA1                            1
CCD                             1
SCA17                           1
HFGS                            1
OPDM1                           1
FAME1                           1
ALS.FTD                         1
FA                              1
HSAN8                           1
OPML1                           1
FRA11B                          1
DRPLA                           1
FRA12A                          1
SCA2                            1
HPE

In [21]:
curr_stats = curr_stats.drop_duplicates(subset=["LocusID"])
curr_stats[curr_stats.disease_associated]

,LocusID,weighted_mean_LPS,weighted_std_LPS,weighted_var_LPS,most_common_motif,num_alleles_max_motif,percent_max_motif,num_total_lps_motifs,num_unique_lps,chrom,start,end,disease_associated,str_period,str_motif,mei,disease_id,GencodeGeneRegion
52485,101068,66.436170,58.214215,3388.894862,AAAAT,188.0,1.000000,1,19,chr1,57367043,57367118,True,5.0,AAAAT,True,SCA37,intron
109796,208930,47.393782,13.511859,182.570324,CCG,193.0,1.000000,1,22,chr1,149390802,149390841,True,3.0,GGC,False,NIID,CDS
271441,511206,62.994350,34.974809,1223.237256,AAAAT,177.0,0.898477,5,16,chr2,96197066,96197121,True,5.0,AAAAT,True,FAME2,intron
274501,517099,28.569231,16.823947,283.045207,CCG,195.0,1.000000,1,24,chr2,100104798,100104822,True,3.0,GCC,False,FRA2A,intron
337137,631340,13.563158,1.498670,2.246011,CCG,190.0,1.000000,1,2,chr2,176093058,176093103,True,3.0,GCG,False,SPD,CDS
348815,652277,38.822581,11.964361,143.145942,AGC,186.0,1.000000,1,18,chr2,190880872,190880920,True,3.0,GCA,False,GAD,5' UTR
443285,828822,30.108808,3.757352,14.117694,AGC,193.0,1.000000,1,8,chr3,63912684,63912714,True,3.0,GCA,False,SCA7,CDS
537980,1004330,45.242424,11.130167,123.880624,AAAAT,165.0,0.873016,2,12,chr3,183712187,183712222,True,5.0,TTTTA,True,FAME4,intron
552781,1033259,53.280000,9.272087,85.971600,AGC,200.0,1.000000,1,17,chr4,3074876,3075040,True,3.0,CAG,False,HD,CDS
583179,1087753,106.006098,132.293534,17501.579231,AAAAG,164.0,0.970414,3,38,chr4,39348424,39348479,True,5.0,AAAAG,True,CANVAS,intron


In [29]:
# save as version 3 so we can use this as the starting point for LPS instead of v2
curr_stats.to_csv("daTR_update_10spanning_curr_stats_v3.csv", index=False)


In [36]:
updated_labels = pd.read_csv("final_manuscript_labels.tsv", sep="\t")
LPS = pd.read_csv("daTR_update_10spanning_curr_stats_v3.csv")
genomic_annotations = pd.read_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/updated_labels_TRID_gene_annotations.csv")
LPS.rename(columns={"weighted_mean_LPS": "Mean", "weighted_std_LPS": "Stdev", 
                    "weighted_var_LPS": "Var","num_unique_lps": "unique_lps"}, inplace=True)
LPS = LPS[["LocusID","chrom","start","end","Mean","Stdev","unique_lps","disease_associated",
           "str_period", "str_motif","most_common_motif","mei","disease_id","num_total_lps_motifs"]].merge(updated_labels[["LocusID","mei","mei_family","mei_subfamily","start_mei","end_mei","internal_external","terminal_label","GencodeGeneRegion"]], on="LocusID", how="left",suffixes=("_orig", ""))
LPS = LPS.drop_duplicates(subset=["LocusID"])
# Merge LPS with genomic_annotations to get the missing GencodeGeneRegion values
LPS = LPS.merge(genomic_annotations[["LocusID", "GencodeGeneRegion"]], on="LocusID", how="left", suffixes=("", "_genomic"))

# Fill missing GencodeGeneRegion values in LPS with the values from genomic_annotations
LPS["GencodeGeneRegion"] = LPS["GencodeGeneRegion"].fillna(LPS["GencodeGeneRegion_genomic"])

# Drop the temporary column used for merging
LPS.drop(columns=["GencodeGeneRegion_genomic"], inplace=True)

# fill the terminal_label column with non_mei if non_mei in mei_orig
LPS.loc[LPS["mei"] == False, "terminal_label"] = "non_mei"
# drop the mei_orig column
LPS = LPS[LPS.terminal_label.notna()]
LPS

/tmp/ipykernel_2365453/2704113908.py:1: DtypeWarning: Columns (4,10) have mixed types. Specify dtype option on import or set low_memory=False.
  updated_labels = pd.read_csv("final_manuscript_labels.tsv", sep="\t")


,LocusID,chrom,start,end,Mean,Stdev,unique_lps,disease_associated,str_period,str_motif,...,disease_id,num_total_lps_motifs,mei,mei_family,mei_subfamily,start_mei,end_mei,internal_external,terminal_label,GencodeGeneRegion
0,15,chr1,14069,14081,12.000000,0.000000,1,False,4.0,CCTC,...,non_disease_associated,1,False,non_mei,non_mei,NaN,NaN,non_mei,non_mei,exon
1,16,chr1,15811,15822,5.887500,1.894028,3,False,3.0,CTG,...,non_disease_associated,1,False,non_mei,non_mei,NaN,NaN,non_mei,non_mei,exon
2,17,chr1,16619,16632,11.944099,0.707093,2,False,3.0,GCT,...,non_disease_associated,1,False,non_mei,non_mei,NaN,NaN,non_mei,non_mei,exon
3,18,chr1,16716,16727,8.981366,0.235698,2,False,3.0,GGT,...,non_disease_associated,1,False,non_mei,non_mei,NaN,NaN,non_mei,non_mei,exon
4,19,chr1,19175,19184,9.000000,0.000000,1,False,3.0,TCC,...,non_disease_associated,1,False,non_mei,non_mei,NaN,NaN,non_mei,non_mei,intron
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2380954,4542948,chr4,3074876,3075040,6.345000,0.957066,2,False,3.0,CCT,...,non_disease_associated,1,False,non_mei,non_mei,NaN,NaN,non_mei,non_mei,CDS
2380955,4542949,chr4,3074876,3075040,9.000000,0.000000,1,False,3.0,CCG,...,non_disease_associated,1,False,non_mei,non_mei,NaN,NaN,non_mei,non_mei,CDS
2380956,4542950,chr4,3074876,3075040,8.969849,0.299238,2,False,3.0,GCC,...,non_disease_associated,1,False,non_mei,non_mei,NaN,NaN,non_mei,non_mei,CDS
2380957,4542951,chr13,70139351,70139428,44.640000,36.198901,18,True,3.0,CTG,...,SCA8,1,False,non_mei,non_mei,NaN,NaN,non_mei,non_mei,CDS


In [37]:
LPS.to_csv("LPS_metrics_with_annotations.tsv.gz", index=False, compression="gzip", sep="\t")

In [35]:
updated_labels

,chr,start,end,LocusID,mei_id,start_mei,end_mei,mei_family,mei_subfamily,orientation,...,mei,terminal_label,GencodeGeneRegion,RefseqGeneRegion,ManeGeneRegion,genomic_annotation,keep_rel_pos,disease_id,disease_associated,simple_motif
0,chr1,26932,26941,26.0,19.0,26791.0,27053.0,SINE/Alu,AluSp,+,...,True,internal,intergenic,intron,intergenic,intergenic,True,non_disease_associated,False,ACC
1,chr1,27863,27872,27.0,21.0,27833.0,28014.0,SINE/MIR,MIRb,C,...,True,internal,intergenic,intron,intergenic,intergenic,True,non_disease_associated,False,ATC
2,chr1,28246,28255,28.0,22.0,28151.0,28302.0,SINE/MIR,MIR,C,...,True,internal,intergenic,intron,intergenic,intergenic,True,non_disease_associated,False,ACT
3,chr1,30867,30892,34.0,26.0,30694.0,30848.0,LTR/ERVL-MaLR,MLT1A,+,...,True,3'terminal,intron,intron,intergenic,intron,True,non_disease_associated,False,AG
4,chr1,30935,30948,36.0,26.0,30953.0,31131.0,LTR/ERVL-MaLR,MLT1A,+,...,True,5'terminal,intron,intron,intergenic,intron,True,non_disease_associated,False,AG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2430050,chrY,56694118,56694136,4542235.0,non_mei,NaN,NaN,non_mei,non_mei,non_mei,...,False,non_mei,intergenic,intergenic,intergenic,intergenic,True,non_disease_associated,False,AATGG
2430051,chrY,56694221,56694236,4542236.0,non_mei,NaN,NaN,non_mei,non_mei,non_mei,...,False,non_mei,intergenic,intergenic,intergenic,intergenic,True,non_disease_associated,False,AATGG
2430052,chrY,56694333,56694350,4542237.0,non_mei,NaN,NaN,non_mei,non_mei,non_mei,...,False,non_mei,intergenic,intergenic,intergenic,intergenic,True,non_disease_associated,False,AATGG
2430053,chrY,56694379,56694395,4542238.0,non_mei,NaN,NaN,non_mei,non_mei,non_mei,...,False,non_mei,intergenic,intergenic,intergenic,intergenic,True,non_disease_associated,False,AATGG
